In [21]:
import pandas as pd
import re
import pyarrow.parquet as pq
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score, accuracy_score,
    precision_recall_fscore_support, classification_report
)

def normalize_cwe(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = s.replace("cwe-", "").replace("cwe ", "").strip()
    m = re.search(r"(\d+)", s)
    return m.group(1) if m else s

def to_binary_series(s):
    s_conv = pd.to_numeric(s, errors="coerce")
    mask_na = s_conv.isna()
    if mask_na.any():
        s_str = s.astype(str).str.strip().str.lower()
        s_conv.loc[mask_na & s_str.isin({"1","true","yes","y","t"})] = 1
        s_conv.loc[mask_na & s_str.isin({"0","false","no","n","f"})] = 0
    s_conv = s_conv.where(s_conv.isin([0, 1]))
    return s_conv

def compute_cwe_metrics(true_cwe, pred_cwe, evaluate_only_with_true=True, zero_division=0):
    """
    Returns:
      {
        "accuracy": float,
        "micro": {"precision":..,"recall":..,"f1":..},
        "macro": {...},
        "weighted": {...},
        "per_class": pd.DataFrame (index=label, cols=["precision","recall","f1","support"]),
        "classification_report_df": pd.DataFrame
      }
    """
    true_cwe = pd.Series(true_cwe).astype(str).fillna("")
    pred_cwe = pd.Series(pred_cwe).astype(str).fillna("")

    if evaluate_only_with_true:
        mask = true_cwe != ""
        true_eval = true_cwe[mask]
        pred_eval = pred_cwe[mask]
    else:
        true_eval = true_cwe
        pred_eval = pred_cwe

    if len(true_eval) == 0:
        return {
            "error": "No rows to evaluate (no ground-truth CWE rows).",
            "accuracy": 0.0,
            "micro": {"precision": 0.0, "recall": 0.0, "f1": 0.0},
            "macro": {"precision": 0.0, "recall": 0.0, "f1": 0.0},
            "weighted": {"precision": 0.0, "recall": 0.0, "f1": 0.0},
            "per_class": pd.DataFrame(),
            "classification_report_df": pd.DataFrame()
        }

    labels = sorted(set(true_eval) | set(pred_eval))

    # micro, macro, weighted
    prec_micro, rec_micro, f1_micro, _ = precision_recall_fscore_support(
        true_eval, pred_eval, average="micro", zero_division=zero_division
    )
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        true_eval, pred_eval, average="macro", zero_division=zero_division
    )
    prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(
        true_eval, pred_eval, average="weighted", zero_division=zero_division
    )

    # per-class breakdown
    p, r, f1, support = precision_recall_fscore_support(
        true_eval, pred_eval, labels=labels, zero_division=zero_division
    )
    per_class_df = pd.DataFrame({
        "precision": p,
        "recall": r,
        "f1": f1,
        "support": support.astype(int)
    }, index=labels)

    report_dict = classification_report(true_eval, pred_eval, labels=labels, zero_division=zero_division, output_dict=True)
    report_df = pd.DataFrame(report_dict).T

    accuracy = accuracy_score(true_eval, pred_eval)

    return {
        "accuracy": float(accuracy),
        "micro": {"precision": float(prec_micro), "recall": float(rec_micro), "f1": float(f1_micro)},
        "macro": {"precision": float(prec_macro), "recall": float(rec_macro), "f1": float(f1_macro)},
        "weighted": {"precision": float(prec_weighted), "recall": float(rec_weighted), "f1": float(f1_weighted)},
        "per_class": per_class_df,
        "classification_report_df": report_df
    }

def evaluate_file(base_df, pred_df):
    # Align indices
    common_idx = base_df.index.intersection(pred_df.index)
    if len(common_idx) == 0:
        raise ValueError("No overlapping indices between base and prediction DataFrames.")

    y_true_series = to_binary_series(base_df.loc[common_idx, "vul"])
    y_pred_series = to_binary_series(pred_df.loc[common_idx, "vul"])
    invalid = y_true_series.isna() | y_pred_series.isna()

    # Keep only valid rows
    valid_idx = y_true_series[~invalid].index
    y_true = y_true_series.loc[valid_idx].astype(int)
    y_pred = y_pred_series.loc[valid_idx].astype(int)

    # Confusion matrix - handle degenerate cases
    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    except ValueError:
        # If only one class present, build manual counts
        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())
        tn = int(((y_pred == 0) & (y_true == 0)).sum())

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)

    # CWE metrics: normalize and restrict to valid rows
    true_cwe = base_df.loc[common_idx, "CWE ID"].apply(normalize_cwe).loc[valid_idx]
    pred_cwe = pred_df.loc[common_idx, "CWE ID"].apply(normalize_cwe).loc[valid_idx]

    # Overall (exact match) accuracy for CWE
    cwe_overall_accuracy = accuracy_score(true_cwe, pred_cwe)

    # Compute multi-class CWE metrics (micro/macro/weighted + per-class)
    cwe_metrics = compute_cwe_metrics(true_cwe, pred_cwe, evaluate_only_with_true=True, zero_division=0)
    # cwe_metrics contains: accuracy, micro/macro/weighted dicts, per_class, classification_report_df

    result = {
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": precision, "recall": recall, "f1": f1, "accuracy": accuracy,
        "cwe_accuracy": cwe_overall_accuracy,
        "cwe_micro_precision": cwe_metrics["micro"]["precision"],
        "cwe_micro_recall": cwe_metrics["micro"]["recall"],
        "cwe_micro_f1": cwe_metrics["micro"]["f1"],
        "cwe_macro_precision": cwe_metrics["macro"]["precision"],
        "cwe_macro_recall": cwe_metrics["macro"]["recall"],
        "cwe_macro_f1": cwe_metrics["macro"]["f1"],
        "cwe_weighted_precision": cwe_metrics["weighted"]["precision"],
        "cwe_weighted_recall": cwe_metrics["weighted"]["recall"],
        "cwe_weighted_f1": cwe_metrics["weighted"]["f1"],
        "evaluated_rows": len(y_true), "dropped_rows": int(invalid.sum()),
        # include the per-class DF so callers can inspect separately
        "_per_class_df": cwe_metrics["per_class"],
        "_classification_report_df": cwe_metrics["classification_report_df"]
    }
    return result

def evaluate_predictions(base_file, pred_files):
    base_df = read_parquet_file(base_file)

    summary_results = {}
    per_class_reports = {}  # file path -> per-class DataFrame
    classification_reports = {}

    for f in pred_files:
        pred_df = read_parquet_file(f)
        res = evaluate_file(base_df, pred_df)

        # Extract per-class and classification report (pop them out of res)
        per_class_df = res.pop("_per_class_df", pd.DataFrame())
        classif_report_df = res.pop("_classification_report_df", pd.DataFrame())

        summary_results[f] = res
        per_class_reports[f] = per_class_df
        classification_reports[f] = classif_report_df

    summary_df = pd.DataFrame(summary_results).T

    # Ensure integer columns remain ints
    for col in ["TP", "FP", "FN", "TN", "evaluated_rows", "dropped_rows"]:
        if col in summary_df.columns:
            summary_df[col] = summary_df[col].astype(int)

    return summary_df, per_class_reports, classification_reports

def read_parquet_file(file_path):
    """
    Reads parquet while removing duplicate __index_level_0__ columns if present.
    Returns a pandas DataFrame.
    """
    parquet_file = pq.ParquetFile(file_path)
    schema = parquet_file.schema
    unique_columns = []
    seen = set()
    for field in schema:
        if field.name not in seen:
            unique_columns.append(field.name)
            seen.add(field.name)
    table = parquet_file.read(columns=unique_columns)
    df = table.to_pandas()

    # Defensive cleanup for duplicated saved index columns
    for idx_col in ["__index_level_0__", "__index_level_0___0"]:
        if idx_col in df.columns:
            df = df.drop(columns=[c for c in df.columns if c == idx_col])

    # Reset index to ensure consistent indexing with base
    df = df.reset_index(drop=True)
    return df


In [32]:
summary_df, per_class_reports, classification_reports = evaluate_predictions(
    "prompts_full_dataset.parquet",
    ["gpt_40_mini_results.parquet", "gpt_41_results.parquet", "llama_3_3_results.parquet", "gpt_3_5_turbo_results.parquet"]
)

# summary table:
print(summary_df)

# pretty-format percentages and integer columns if you like:
styled = summary_df.copy()
for col in ["precision","recall","f1","accuracy","cwe_accuracy",
            "cwe_micro_precision","cwe_micro_recall","cwe_micro_f1",
            "cwe_macro_precision","cwe_macro_recall","cwe_macro_f1",
            "cwe_weighted_precision","cwe_weighted_recall","cwe_weighted_f1"]:
    if col in styled.columns:
        styled[col] = styled[col].map(lambda x: f"{x:.2%}")
for col in ["TP","FP","FN","TN","evaluated_rows","dropped_rows"]:
    if col in styled.columns:
        styled[col] = styled[col].astype(int)
# Show as styled table (works in Jupyter/Colab)
styled = styled.style.set_caption("Evaluation Results").set_table_styles(
    [{'selector': 'th', 'props': [('text-align', 'center')]},
     {'selector': 'td', 'props': [('text-align', 'center')]}]
)
styled

                                TP   FP   FN    TN  precision    recall  \
gpt_40_mini_results.parquet    698  839  175   541   0.454131  0.799542   
gpt_41_results.parquet         502  339  367  1034   0.596908  0.577675   
llama_3_3_results.parquet      495  447  378   933   0.525478  0.567010   
gpt_3_5_turbo_results.parquet  355  380  515   993   0.482993  0.408046   

                                     f1  accuracy  cwe_accuracy  \
gpt_40_mini_results.parquet    0.579253  0.549933      0.407013   
gpt_41_results.parquet         0.587135  0.685103      0.613738   
llama_3_3_results.parquet      0.545455  0.633822      0.536618   
gpt_3_5_turbo_results.parquet  0.442368  0.600981      0.525635   

                               cwe_micro_precision  cwe_micro_recall  \
gpt_40_mini_results.parquet               0.407013          0.407013   
gpt_41_results.parquet                    0.613738          0.613738   
llama_3_3_results.parquet                 0.536618          0.536618   


,TP,FP,FN,TN,precision,recall,f1,accuracy,cwe_accuracy,cwe_micro_precision,cwe_micro_recall,cwe_micro_f1,cwe_macro_precision,cwe_macro_recall,cwe_macro_f1,cwe_weighted_precision,cwe_weighted_recall,cwe_weighted_f1,evaluated_rows,dropped_rows
gpt_40_mini_results.parquet,698,839,175,541,45.41%,79.95%,57.93%,54.99%,40.70%,40.70%,40.70%,40.70%,3.83%,4.41%,3.92%,57.82%,40.70%,45.02%,2253,0
gpt_41_results.parquet,502,339,367,1034,59.69%,57.77%,58.71%,68.51%,61.37%,61.37%,61.37%,61.37%,25.28%,21.50%,22.87%,61.75%,61.37%,61.28%,2242,11
llama_3_3_results.parquet,495,447,378,933,52.55%,56.70%,54.55%,63.38%,53.66%,53.66%,53.66%,53.66%,6.07%,4.89%,5.31%,57.65%,53.66%,55.29%,2253,0
gpt_3_5_turbo_results.parquet,355,380,515,993,48.30%,40.80%,44.24%,60.10%,52.56%,52.56%,52.56%,52.56%,6.98%,4.69%,5.48%,53.06%,52.56%,52.14%,2243,10


In [33]:
import os
import re
from pathlib import Path

def _sanitize_name(s, maxlen=60):
    s = re.sub(r'[\[\]\*:/\\\?\']', '_', str(s))
    return s[:maxlen]

def save_summary_csv(
    summary_df,
    per_class_reports=None,
    out_dir="eval_output",
    summary_fname="evaluation_results_summary.csv",
    save_per_class=True,
    pct_cols=None,
    int_cols=None,
):
    """
    Save formatted summary_df to CSV and optional per-class CSVs.

    Parameters
    ----------
    summary_df : pd.DataFrame
        Raw numeric summary returned by evaluate_predictions().
    per_class_reports : dict or None
        Optional dict mapping model_filename -> per-class DataFrame.
    out_dir : str
        Directory to save CSV files.
    summary_fname : str
        Filename for the summary CSV.
    save_per_class : bool
        Whether to save per-class dataframes to out_dir/per_class/.
    pct_cols : list or None
        Columns to format as percentages. Defaults to common metric names.
    int_cols : list or None
        Columns to force to int. Defaults to common count columns.
    """
    import pandas as pd
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # sensible defaults
    if pct_cols is None:
        pct_cols = [
            "precision","recall","f1","accuracy","cwe_accuracy",
            "cwe_micro_precision","cwe_micro_recall","cwe_micro_f1",
            "cwe_macro_precision","cwe_macro_recall","cwe_macro_f1",
            "cwe_weighted_precision","cwe_weighted_recall","cwe_weighted_f1"
        ]
    if int_cols is None:
        int_cols = ["TP","FP","FN","TN","evaluated_rows","dropped_rows"]

    df = summary_df.copy()

    # Format percentage columns as "59.69%" strings (skip missing)
    for c in pct_cols:
        if c in df.columns:
            df[c] = df[c].map(lambda x: f"{x:.2%}" if pd.notna(x) else "")

    # Force integer-like count columns to int (if present)
    for c in int_cols:
        if c in df.columns:
            # Sometimes column may already be int dtype; safe-cast
            try:
                df[c] = df[c].astype(int)
            except Exception:
                # If some NaNs present, fill with 0 then cast
                df[c] = df[c].fillna(0).astype(int)

    # Save the summary CSV
    summary_path = out_dir / summary_fname
    df.to_csv(summary_path, index=True)
    print(f"Saved summary CSV: {summary_path.resolve()}")

    # Optionally save per-class reports
    if save_per_class and per_class_reports:
        per_dir = out_dir / "per_class"
        per_dir.mkdir(exist_ok=True)
        for fname, pc_df in per_class_reports.items():
            safe = _sanitize_name(fname, maxlen=80)
            out_path = per_dir / f"{safe}_per_class.csv"
            # Convert index (label) to column if it's the index for readability
            try:
                to_save = pc_df.copy()
                if to_save.index.name is None:
                    to_save = to_save.reset_index().rename(columns={"index": "label"})
                to_save.to_csv(out_path, index=False)
            except Exception:
                # fallback: save with index
                pc_df.to_csv(out_path, index=True)
            print(f"Saved per-class CSV: {out_path.resolve()}")

    return {"summary_csv": str(summary_path.resolve())}


In [35]:
summary_df, per_class_reports, classification_reports = evaluate_predictions(
    "prompts_full_dataset.parquet",
    ["gpt_40_mini_results.parquet","gpt_41_results.parquet","llama_3_3_results.parquet","gpt_3_5_turbo_results.parquet"]
)

result_paths = save_summary_csv(summary_df, per_class_reports, out_dir="eval_output", summary_fname="evaluation_results_summary.csv")
print(result_paths)

Saved summary CSV: F:\Projects\SecRAG\eval_output\evaluation_results_summary.csv
Saved per-class CSV: F:\Projects\SecRAG\eval_output\per_class\gpt_40_mini_results.parquet_per_class.csv
Saved per-class CSV: F:\Projects\SecRAG\eval_output\per_class\gpt_41_results.parquet_per_class.csv
Saved per-class CSV: F:\Projects\SecRAG\eval_output\per_class\llama_3_3_results.parquet_per_class.csv
Saved per-class CSV: F:\Projects\SecRAG\eval_output\per_class\gpt_3_5_turbo_results.parquet_per_class.csv
{'summary_csv': 'F:\\Projects\\SecRAG\\eval_output\\evaluation_results_summary.csv'}
